# Bronze to Silver
- Limpeza e padronização dos dados, assim como aplicação das regras de negócio

## 1. Configurações Iniciais

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

print(f"Schema {CATALOG}.{SILVER} criado ou já existente.")

Schema workspace.silver criado ou já existente.


In [0]:
#Silver reconstruida com "overwrite", p/ não duplicar os dados já tratados
def save_silver(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SILVER}.{table_name}")
    )

    print(f"{CATALOG}.{SILVER}.{table_name} criada com sucesso.")

##### Função para manter a mais recente

In [0]:
def keep_latest_by_id(df, id_col="id"):
    # Mantém 1 registro por filme: o mais recente por ingestion_datetime
    # Ids nulos/em branco são descartados
    data_cols = [c for c in df.columns if c != "ingestion_datetime"]

    completude = sum(
        (F.when(F.trim(F.col(c).cast("string")) != "", 1).otherwise(0) for c in data_cols),
        F.lit(0)
    )
    hash_linha = F.sha2(
        F.concat_ws("||", *[F.col(c).cast("string") for c in data_cols]), 256
    )

    w = (
        Window
        .partitionBy(id_col)
        .orderBy(
            F.col("ingestion_datetime").desc_nulls_last(),
            completude.desc(),
            hash_linha.desc()
        )
    )

    return (
        df
        .filter(F.col(id_col).isNotNull() & (F.trim(F.col(id_col).cast("string")) != ""))
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


## 2. silver.tb_info_filmes
- Mapeamento de colunas
- Limpeza e tradução do status
- Deduplicação
- Tratamento de data
- Criação de coluna ano_lancamento

#### 2.1 Função de tratamento das datas

In [0]:
FORMATOS_DATA = [
    "d/M/yyyy", "M/d/yyyy", "yyyy/M/d",
    "d-M-yyyy", "M-d-yyyy", "yyyyMMdd",
    "MMMM d, yyyy", "MMM d, yyyy", "d MMM yyyy", "d MMMM yyyy",
]

def date_multi_format(col_name):
    c = f"`{col_name}`"

    return F.coalesce(
        F.expr(f"try_cast(trim({c}) AS DATE)"),
        *[
            F.expr(f"CAST(try_to_timestamp(trim({c}), '{fmt}') AS DATE)")
            for fmt in FORMATOS_DATA
        ]
    )


#### 2.2 Deduplicação e tratamento do "Status"
- Remove espaços extras
- Substitui hífens
- Padroniza para minusculas

In [0]:
df_info = spark.table(
    f"{CATALOG}.{BRONZE}.tb_movies_info"
)

#Mantem somente a versão mais recente de cada filme
df_info = keep_latest_by_id(df_info)

df_info = df_info.withColumn(
    "_status_normalizado",
    F.lower(
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.col("status"),
                    r"[-_]+",
                    " "
                ),
                r"[^A-Za-zÀ-ÿ ]",
                ""
            )
        )
    )
)

df_info = df_info.withColumn(
    "_status_normalizado",
    F.regexp_replace(
        F.col("_status_normalizado"),
        r"\s+",
        " "
    )
)

status_map = (
    F.when(F.col("_status_normalizado") == "released", "Lançado")
     .when(F.col("_status_normalizado") == "post production", "Pós-Produção")
     .when(F.col("_status_normalizado") == "in production", "Em Produção")
     .when(F.col("_status_normalizado") == "planned", "Planejado")
     .when(F.col("_status_normalizado") == "rumored", "Rumores")
     .when(
         F.col("_status_normalizado").isin("canceled", "cancelled"),
         "Cancelado"
     )
     .otherwise("Não Informado")
)

runtime_clean = F.trim(F.col("runtime"))

runtime_raw = (
    F.when(
        runtime_clean.rlike(r"^\d{1,5}([.,]0+)?$"),   # até 5 dígitos: nunca estoura o INT
        F.regexp_replace(
            runtime_clean,
            r"[.,]0+$",
            ""
        ).cast("int")
    )
    .otherwise(F.lit(None).cast("int"))
)

runtime_int = F.when(runtime_raw > 0, runtime_raw)

df_info_silver = (
    df_info
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.trim(F.col("title")).alias("titulo"),
        F.trim(F.col("original_title")).alias("titulo_original"),
        date_multi_format("release_date").alias("data_lancamento"),
        runtime_int.alias("duracao_minutos"),
        F.trim(F.col("original_language")).alias("idioma_original"),
        status_map.alias("status_filme"),
        F.trim(F.col("overview")).alias("sinopse"),
        F.trim(F.col("tagline")).alias("frase_divulgacao")
    )
    .withColumn(
        "ano_lancamento",
        F.year("data_lancamento")
    )
)

save_silver(df_info_silver, "tb_info_filmes")

workspace.silver.tb_info_filmes criada com sucesso.


In [0]:
display(df_info_silver)
df_info_silver.printSchema()

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On va manquer !,On va manquer !,2018-05-15,null,fr,Lançado,null,null,2018
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,null,es,Lançado,null,null,2021
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null,2022
1000058,Homecoming,Le retour,2023-07-12,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null,2023
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null,2016
1000073,A Chance To Win,Pour l'honneur,2023-05-03,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null,2023


root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)



## 3. silver.tb_cotacao_dolar
- Estruturar o histórico para garantir uma série temporal contínua
- Aplicar Forward Fill para dias sem cotação

#### 3.1 Estruturação

In [0]:
df_cotacao = spark.table(
    f"{CATALOG}.{BRONZE}.tb_cotacao_dolar"
)

df_cotacao = (
    df_cotacao
    .withColumn(
        "timestamp_cotacao",
        F.coalesce(
            F.expr(
                "try_cast(dataHoraCotacao AS TIMESTAMP)"
            ),
            F.expr(
                """
                try_to_timestamp(
                    dataHoraCotacao,
                    'yyyy-MM-dd HH:mm:ss.SSS'
                )
                """
            )
        )
    )
    .withColumn(
        "data_cotacao",
        F.to_date("timestamp_cotacao")
    )
    .withColumn(
        "cotacao_compra",
        F.col("cotacaoCompra").cast("decimal(18,6)")
    )
    .filter(
        F.col("data_cotacao").isNotNull()
        & F.col("cotacao_compra").isNotNull()
    )
)

In [0]:
#Pegar cotação mais recente de cada dia
w_cotacao_dia = (
    Window
    .partitionBy("data_cotacao")
    .orderBy(
        F.col("timestamp_cotacao").desc(),
        F.col("ingestion_datetime").desc()
    )
)

df_cotacao_diaria = (
    df_cotacao
    .withColumn(
        "_rn",
        F.row_number().over(w_cotacao_dia)
    )
    .filter(F.col("_rn") == 1)
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
)

##### Criação do calendário

In [0]:
limites = (
    df_cotacao_diaria
    .agg(
        F.min("data_cotacao").alias("data_min"),
        F.max("data_cotacao").alias("data_max")
    )
    .first()
)

if limites["data_min"] is None:
    raise Exception(
        "Não existem cotações válidas na camada Bronze."
    )

df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(limites["data_min"]),
                # até hoje: fins de semana/feriados no fim da série também recebem a última cotação
                F.greatest(F.lit(limites["data_max"]), F.current_date()),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

#### 3.2 Forward Fill

In [0]:
w_forward_fill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

df_cotacao_silver = (
    df_calendario
    .join(
        df_cotacao_diaria,
        "data_cotacao",
        "left"
    )
    .withColumn(
        "cotacao_preenchida",
        F.col("cotacao_compra").isNull()   # True = dia sem cotação, preenchido (forward fill)
    )
    .withColumn(
        "cotacao_compra",
        F.last(
            "cotacao_compra",
            ignorenulls=True
        ).over(w_forward_fill)
    )
)

display(df_cotacao_silver)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-14,5.169000,false
2026-09-15,5.148400,false
2026-09-16,5.152000,false
2026-09-17,5.151500,false
2026-09-18,5.156900,false
2026-09-19,5.156900,true
2026-09-20,5.156900,true
2026-09-21,5.156900,true


In [0]:
save_silver(
    df_cotacao_silver,
    "tb_cotacao_dolar"
)

workspace.silver.tb_cotacao_dolar criada com sucesso.


## 4. silver.tb_financeiro_filmes
- Trata valores que representam ausência
- Higieniza colunas de orçamento e converte métricas para o tipo numérico apropriado
- Calcula valor equivalente em Reais(BRL)

##### Função para remover símbolos de moeda e textos "NULL"

In [0]:
NULL_VALUES = [
    "",
    "unknown",
    "não informado",
    "nao informado",
    "null",
    "none",
    "n/a",
    "na",
    "-"
]

def clean_money(col_name):
    raw = F.trim(F.col(col_name).cast("string"))

    raw = (
        F.when(
            raw.isNull()
            | F.lower(raw).isin(NULL_VALUES),
            F.lit(None)
        )
        .otherwise(raw)
    )

    # remove símbolos/códigos de moeda e rejeita qualquer texto restante
    raw = F.regexp_replace(raw, r"(?i)(USD|BRL|US\$|R\$|\$)", "")
    raw = F.when(raw.rlike(r"[A-Za-z]"), F.lit(None)).otherwise(F.trim(raw))

    #deixa somente numeros, sinal, ponto e virgula
    clean = F.regexp_replace(
        raw,
        r"[^0-9,.\-]",
        ""
    )

    has_comma = clean.contains(",")
    has_dot = clean.contains(".")
    both = has_comma & has_dot

    br_format = (
        F.regexp_replace(
            F.regexp_replace(clean, r"\.", ""),
            ",",
            "."
        )
    )

    us_format = F.regexp_replace(
        clean,
        ",",
        ""
    )

    comma_decimal = F.regexp_replace(
        clean,
        ",",
        "."
    )

    normalized = (
        F.when(
            both & clean.rlike(r",\d{1,2}$"),
            br_format
        )
        .when(
            both,
            us_format
        )
        .when(
            has_comma & clean.rlike(r"^-?\d+,\d{1,2}$"),
            comma_decimal
        )
        .when(
            has_comma,
            F.regexp_replace(clean, ",", "")
        )
        .when(
            clean.rlike(r"^-?\d{1,3}(\.\d{3})+$"),
            F.regexp_replace(clean, r"\.", "")
        )
        .otherwise(clean)
    )

    return (
        F.when(
            normalized.rlike(r"^-?\d{1,13}(\.\d+)?$"),
            normalized.cast("decimal(18,2)")
        )
        .otherwise(
            F.lit(None).cast("decimal(18,2)")
        )
    )

In [0]:
#Pegar cotação mais recente
cotacao_mais_recente = (
    spark.table(
        f"{CATALOG}.{SILVER}.tb_cotacao_dolar"
    )
    .filter(F.col("cotacao_compra").isNotNull())
    .orderBy(F.col("data_cotacao").desc())
    .select("cotacao_compra")
    .first()
)

if cotacao_mais_recente is None:
    raise Exception(
        "Nenhuma cotação válida encontrada."
    )

cotacao_atual = cotacao_mais_recente["cotacao_compra"]

print(f"Cotação utilizada: {cotacao_atual}")

Cotação utilizada: 5.156900


In [0]:
df_financeiro = spark.table(
    f"{CATALOG}.{BRONZE}.tb_movies_financials"
)

df_financeiro = keep_latest_by_id(
    df_financeiro
)

df_financeiro = (
    df_financeiro
    .withColumn(
        "_orcamento",
        clean_money("budget")
    )
    .withColumn(
        "_receita",
        clean_money("revenue")
    )
)

df_financeiro = (
    df_financeiro
    .withColumn(
        "orcamento_usd",
        F.when(
            F.col("_orcamento") > 0,
            F.col("_orcamento")
        ).otherwise(
            F.lit(None).cast("decimal(18,2)")
        )
    )
    .withColumn(
        "receita_usd",
        F.when(
            F.col("_receita") > 0,
            F.col("_receita")
        ).otherwise(
            F.lit(None).cast("decimal(18,2)")
        )
    )
)

In [0]:
#dolar p/ real
taxa = F.lit(cotacao_atual).cast(
    "decimal(18,6)"
)

df_financeiro_silver = (
    df_financeiro
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "orcamento_usd",
        "receita_usd"
    )
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * taxa)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * taxa)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_usd",
        (
            F.col("receita_usd")
            - F.col("orcamento_usd")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (
            F.col("receita_brl")
            - F.col("orcamento_brl")
        ).cast("decimal(18,2)")
    )
)

##### Margem de lucro


In [0]:
df_financeiro_silver = (
    df_financeiro_silver
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull()
            & (F.col("receita_usd") != 0),
            F.round(
                (
                    F.col("lucro_usd").cast("double")
                    / F.col("receita_usd").cast("double")
                ) * 100,
                2
            )
        )
        .otherwise(None)
        .cast("decimal(18,2)")
    )
)

save_silver(
    df_financeiro_silver,
    "tb_financeiro_filmes"
)

workspace.silver.tb_financeiro_filmes criada com sucesso.


## 5. silver.tb_metricas_engajamento
- Garante que os valores não sejam invalidos ou convertidos p/ nulo
- Conversão de tipagem para valores decimais e inteiros
- Trata inconsistências de escala e limites numéricos de negócio

#### 5.1 Funções para as métricas

In [0]:
def normalize_decimal_string(col_name):
    raw = F.trim(F.col(col_name).cast("string"))
    raw = F.when(raw.isNull() | F.lower(raw).isin(NULL_VALUES), F.lit(None)).otherwise(raw)

    # Só aceita dígitos, sinal, ponto e vírgula. Qualquer letra/símbolo (ex.: "tt1234567", "7 stars",
    # "8/10") vira NULL em vez de "extrair dígitos" de um valor deslocado (column shift).
    raw = F.when(raw.rlike(r"^-?[0-9.,]+$"), raw)

    # O ÚLTIMO separador é o decimal; os anteriores são milhar. Cobre 12,5 / 1,234.56 / 1.234,56 / 1.234.567
    return F.regexp_replace(
        F.regexp_replace(raw, ",", "."),
        r"\.(?=.*\.)",
        ""
    )


def safe_double(col):
    return (
        F.when(
            col.rlike(r"^-?\d{1,15}(\.\d+)?$"),
            col.cast("double")
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )


In [0]:
def safe_integer(col_name):
    raw = F.trim(
        F.col(col_name).cast("string")
    )

    clean = F.regexp_replace(
        raw,
        r"\s+",
        ""
    )

    normalized = (
        F.when(
            clean.rlike(r"^-?\d+$"),
            clean
        )
        .when(
            clean.rlike(
                r"^-?\d{1,3}([.,]\d{3})+$"
            ),
            F.regexp_replace(
                clean,
                r"[.,]",
                ""
            )
        )
        .when(
            clean.rlike(
                r"^-?\d+[.,]0+$"
            ),
            F.regexp_replace(
                clean,
                r"[.,]0+$",
                ""
            )
        )
        .otherwise(None)
    )

    return F.when(normalized.rlike(r"^-?\d{1,9}$"), normalized.cast("int"))

#### 5.2 Criação da tabela

In [0]:
df_metricas = spark.table(
    f"{CATALOG}.{BRONZE}.tb_movies_metrics"
)

df_metricas = keep_latest_by_id(
    df_metricas
)

df_metricas = (
    df_metricas
    .withColumn(
        "_popularidade",
        safe_double(
            normalize_decimal_string(
                "popularity"
            )
        )
    )
    .withColumn(
        "_nota_tmdb",
        safe_double(
            normalize_decimal_string(
                "vote_average"
            )
        )
    )
    .withColumn(
        "_nota_imdb",
        safe_double(
            normalize_decimal_string(
                "averageRating"
            )
        )
    )
    .withColumn(
        "_votos_tmdb",
        safe_integer("vote_count")
    )
    .withColumn(
        "_votos_imdb",
        safe_integer("numVotes")
    )
)

##### Regras de intervalo

In [0]:
df_metricas_silver = (
    df_metricas
    .select(
        F.col("id").cast("string").alias(
            "id_filme"
        ),

        F.when(
            F.col("_popularidade") >= 0,
            F.col("_popularidade")
        ).otherwise(None).alias(
            "popularidade"
        ),

        F.when(
            F.col("_nota_tmdb").between(
                0, 10
            ),
            F.col("_nota_tmdb")
        ).otherwise(None).alias(
            "nota_media_tmdb"
        ),

        F.when(
            F.col("_votos_tmdb") >= 0,
            F.col("_votos_tmdb")
        ).otherwise(None).alias(
            "qtd_votos_tmdb"
        ),

        F.when(
            F.col("_nota_imdb").between(
                0, 10
            ),
            F.col("_nota_imdb")
        ).otherwise(None).alias(
            "nota_media_imdb"
        ),

        F.when(
            F.col("_votos_imdb") >= 0,
            F.col("_votos_imdb")
        ).otherwise(None).alias(
            "qtd_votos_imdb"
        )
    )
)

save_silver(
    df_metricas_silver,
    "tb_metricas_engajamento"
)

workspace.silver.tb_metricas_engajamento criada com sucesso.


## 6. silver.tb_avaliacoes_usuarios
- Deduplicação
- Garante que a regra de negócio é respeitada (escala permitida de 0 a 10)
- Identifica comentários não preenchidos e padroniza

In [0]:
df_reviews = spark.table(
    f"{CATALOG}.{BRONZE}.tb_movies_reviews"
)

nota_normalizada = safe_double(
    normalize_decimal_string("nota")
)

df_reviews_silver = (
    df_reviews
    .select(
        F.col("id").cast("string").alias(
            "id_filme"
        ),

        F.trim(F.col("nome")).alias(
            "nome_usuario"
        ),

        F.when(
            nota_normalizada.between(0, 10),
            nota_normalizada
        ).otherwise(None).alias(
            "nota_usuario"
        ),

        F.when(
            F.col("comentario").isNull()
            | ~F.col("comentario").rlike(r"[^\s\u00a0]"),
            F.lit("Sem comentário")
        )
        .otherwise(
            F.regexp_replace(
                F.col("comentario"),
                r"^[\s\u00a0]+|[\s\u00a0]+$",
                ""
            )
        )
        .alias("comentario_usuario")
    )
    .dropDuplicates([
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    ])
)

save_silver(
    df_reviews_silver,
    "tb_avaliacoes_usuarios"
)

workspace.silver.tb_avaliacoes_usuarios criada com sucesso.


## 7. silver.tb_generos
- Remove resíduos que não pertençam ao domínio de gêneros
- (split + explode) a coluna "genres"

In [0]:
df_credits = keep_latest_by_id(
    spark.table(
        f"{CATALOG}.{BRONZE}.tb_credits_and_tags"
    )
)

generos_validos = {
    "action": "Action",
    "adventure": "Adventure",
    "animation": "Animation",
    "comedy": "Comedy",
    "crime": "Crime",
    "documentary": "Documentary",
    "drama": "Drama",
    "family": "Family",
    "fantasy": "Fantasy",
    "history": "History",
    "horror": "Horror",
    "music": "Music",
    "mystery": "Mystery",
    "romance": "Romance",
    "science fiction": "Science Fiction",
    "tv movie": "TV Movie",
    "thriller": "Thriller",
    "war": "War",
    "western": "Western"
}

genre_map = F.create_map(
    *[
        item
        for k, v in generos_validos.items()
        for item in (
            F.lit(k),
            F.lit(v)
        )
    ]
)

##### Normalização

In [0]:
df_generos = (
    df_credits
    .select(
        F.col("id").cast("string").alias(
            "id_filme"
        ),

        F.explode(
            F.split(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("genres"),
                        ";",
                        ","
                    ),
                    r"""[\[\]"']""",
                    ""
                ),
                r"\s*,\s*"
            )
        ).alias("_genero")
    )
    .withColumn(
        "_genero",
        F.lower(
            F.trim(
                F.col("_genero")
            )
        )
    )
    .withColumn(
        "nome_genero",
        genre_map[
            F.col("_genero")
        ]
    )
    .filter(
        F.col("nome_genero").isNotNull()
    )
    .select(
        "id_filme",
        "nome_genero"
    )
    .dropDuplicates()
)

save_silver(
    df_generos,
    "tb_generos"
)

workspace.silver.tb_generos criada com sucesso.


## 8. silver.tb_pessoas_empresas
- Consolida 4 tipos de entidade em uma única tabela
- Padroniza a formatação do texto e elimina registros duplicados

In [0]:
def explode_entidade(
    df,
    source_col,
    tipo_entidade,
    max_palavras=6
):
    return (
        df
        .select(
            F.col("id")
            .cast("string")
            .alias("id_filme"),

            F.posexplode(
                F.split(
                    F.regexp_replace(
                        F.regexp_replace(
                            F.col(source_col),
                            ";",
                            ","
                        ),
                        r"""[\[\]"']""",
                        ""
                    ),
                    r"\s*,\s*"
                )
            ).alias("_pos", "_nome")
        )
        .withColumn(
            "_nome",
            F.trim(
                F.col("_nome")
            )
        )
        .filter(
            F.col("_nome").isNotNull()
            & (F.length("_nome") > 0)
        )

        .filter(
            ~F.col("_nome").rlike(
                r"^[0-9.,\-]+$"
            )
        )

        .filter(
            F.col("_nome").rlike(
                r".*[A-Za-zÀ-ÿ].*"
            )
        )

        # Elimina fragmentos de texto
        .filter(F.length("_nome") <= 80)
        .filter(F.size(F.split(F.col("_nome"), r"\s+")) <= max_palavras)
        .filter(~F.col("_nome").contains("\\"))

        .withColumn(
            "nome_entidade",
            F.initcap(
                F.lower(
                    F.col("_nome")
                )
            )
        )

        .withColumn(
            "tipo_entidade",
            F.lit(tipo_entidade)
        )

        .select(
            "id_filme",
            "nome_entidade",
            "tipo_entidade",
            F.col("_pos").alias("ordem_credito")
        )
    )


#### Unir tudo

In [0]:
df_atores = explode_entidade(
    df_credits,
    "cast",
    "Ator"
)

df_diretores = explode_entidade(
    df_credits,
    "directors",
    "Diretor"
)

df_roteiristas = explode_entidade(
    df_credits,
    "writers",
    "Roteirista"
)

df_produtoras = explode_entidade(
    df_credits,
    "production_companies",
    "Produtora",
    max_palavras=8   # nomes de produtoras costumam ser mais longos que nomes de pessoas
)

df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .groupBy("id_filme", "nome_entidade", "tipo_entidade")
    .agg(F.min("ordem_credito").alias("ordem_credito"))
)

save_silver(
    df_pessoas_empresas,
    "tb_pessoas_empresas"
)

workspace.silver.tb_pessoas_empresas criada com sucesso.


In [0]:
display(
    df_pessoas_empresas
    .orderBy(
        F.desc(
            F.length("nome_entidade")
        )
    )
    .limit(100)
)

id_filme,nome_entidade,tipo_entidade,ordem_credito
393184,Ministère Des Affaires Étrangères Et Du Développement International,Produtora,0
603225,Instituto Cubano Del Arte E Industrias Cinematográficos (icaic),Produtora,1
887723,Communication University Of China Television Production Centre,Produtora,3
494401,Saraschandrikaa Visionary Motion Pictures - Mayabazar Pictures,Produtora,0
452557,Beijing Dongfang International Cultural Communications Company,Produtora,1
689766,Centro Regional De Formación Docente E Investigación Educativa,Produtora,0
926935,Państwowa Wyższa Szkoła Filmowa Telewizyjna I Teatralna (łódź),Produtora,0
1155720,Swinburne Institute Of Technology. Film And Television School,Produtora,2
486465,National Film-studio “kyrgyzfilm” Named After Tolomush Okeyev,Produtora,0
1211221,National Association Of Latino Independent Producers (nalip),Produtora,0


### Auditoria de qualidade (informativa)
- Gêneros descartados pela lista de domínio
- Valores preenchidos na origem que viraram NULL
- Ids sem correspondência em `tb_info_filmes`
- Duplicatas reais em avaliações

In [0]:
display(
    df_credits
    .select(
        F.explode(
            F.split(
                F.regexp_replace(
                    F.regexp_replace(F.col("genres"), ";", ","),
                    r"""[\[\]"']""",
                    ""
                ),
                r"\s*,\s*"
            )
        ).alias("_genero")
    )
    .withColumn("_genero", F.lower(F.trim("_genero")))
    .filter(genre_map[F.col("_genero")].isNull())
    .groupBy("_genero").count()
    .orderBy(F.desc("count"))
    .limit(40)
)

def auditar_descartes(bronze_tbl, silver_tbl, mapa):
    b = keep_latest_by_id(spark.table(f"{CATALOG}.{BRONZE}.{bronze_tbl}"))
    s = spark.table(f"{CATALOG}.{SILVER}.{silver_tbl}")
    for col_b, col_s in mapa.items():
        origem = b.filter(F.trim(F.col(col_b)) != "").count()
        valido = s.filter(F.col(col_s).isNotNull()).count()
        print(f"{silver_tbl}.{col_s:<18} origem preenchida: {origem:>8} | válida: {valido:>8} | descartada: {origem - valido:>7}")

auditar_descartes("tb_movies_info", "tb_info_filmes",
                  {"release_date": "data_lancamento", "runtime": "duracao_minutos"})
auditar_descartes("tb_movies_financials", "tb_financeiro_filmes",
                  {"budget": "orcamento_usd", "revenue": "receita_usd"})
auditar_descartes("tb_movies_metrics", "tb_metricas_engajamento",
                  {"popularity": "popularidade", "vote_average": "nota_media_tmdb",
                   "vote_count": "qtd_votos_tmdb", "averageRating": "nota_media_imdb",
                   "numVotes": "qtd_votos_imdb"})

ids_info = spark.table(f"{CATALOG}.{SILVER}.tb_info_filmes").select("id_filme")
for t in ["tb_financeiro_filmes", "tb_metricas_engajamento", "tb_avaliacoes_usuarios"]:
    orfaos = spark.table(f"{CATALOG}.{SILVER}.{t}").join(ids_info, "id_filme", "left_anti").count()
    print(f"{t}: {orfaos} registros com id_filme ausente em tb_info_filmes")

r = spark.table(f"{CATALOG}.{BRONZE}.tb_movies_reviews")
print("Avaliações duplicadas na origem:",
      r.count() - r.dropDuplicates(["id", "nome", "nota", "comentario"]).count())


_genero,count
comedy|drama,199
0.6,172
drama|comedy,124
drama|romance,109
horror|thriller,100
drama|thriller,99
comedy|romance,82
documentary|music,75
documentary|history,50
horror|comedy,50


tb_info_filmes.data_lancamento    origem preenchida:    97823 | válida:    97815 | descartada:       8
tb_info_filmes.duracao_minutos    origem preenchida:    97821 | válida:    87635 | descartada:   10186
tb_financeiro_filmes.orcamento_usd      origem preenchida:    99006 | válida:     7753 | descartada:   91253
tb_financeiro_filmes.receita_usd        origem preenchida:    99006 | válida:     3285 | descartada:   95721
tb_metricas_engajamento.popularidade       origem preenchida:    99012 | válida:    94845 | descartada:    4167
tb_metricas_engajamento.nota_media_tmdb    origem preenchida:    98995 | válida:    95488 | descartada:    3507
tb_metricas_engajamento.qtd_votos_tmdb     origem preenchida:    91507 | válida:    91362 | descartada:     145
tb_metricas_engajamento.nota_media_imdb    origem preenchida:    88810 | válida:    86195 | descartada:    2615
tb_metricas_engajamento.qtd_votos_imdb     origem preenchida:    90956 | válida:    87769 | descartada:    3187
tb_financeiro_fi

## 9. Validações Finais

In [0]:
expected_silver_tables = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

silver_tables = [
    table.name
    for table in spark.catalog.listTables(
        f"{CATALOG}.{SILVER}"
    )
]

faltando = [t for t in expected_silver_tables if t not in silver_tables]

for table in expected_silver_tables:
    if table in silver_tables:
        count = spark.table(
            f"{CATALOG}.{SILVER}.{table}"
        ).count()

        print(
            f"OK - {table}: {count} registros"
        )

    else:
        print(
            f"ERRO - {table} não encontrada"
        )

if faltando:
    raise Exception(f"Tabelas Silver ausentes: {faltando}")


OK - tb_info_filmes: 97879 registros
OK - tb_financeiro_filmes: 99006 registros
OK - tb_metricas_engajamento: 99013 registros
OK - tb_avaliacoes_usuarios: 32412 registros
OK - tb_generos: 132598 registros
OK - tb_pessoas_empresas: 896585 registros
OK - tb_cotacao_dolar: 8 registros


##### Filmes duplicados


In [0]:
display(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_info_filmes"
    )
    .groupBy("id_filme")
    .count()
    .filter("count > 1")
)

id_filme,count


##### Notas inválidas deveriam retornar zero registros


In [0]:
display(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_metricas_engajamento"
    )
    .filter(
        """
        nota_media_tmdb < 0
        OR nota_media_tmdb > 10
        OR nota_media_imdb < 0
        OR nota_media_imdb > 10
        """
    )
)

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb


##### Financeiro com valores <= 0 deveria retornar zero registros

In [0]:
display(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_financeiro_filmes"
    )
    .filter(
        """
        orcamento_usd <= 0
        OR receita_usd <= 0
        """
    )
)

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual


##### Série da cotação


In [0]:
display(
    spark.table(
        f"{CATALOG}.{SILVER}.tb_cotacao_dolar"
    )
    .orderBy("data_cotacao")
)

data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-14,5.169000,false
2026-09-15,5.148400,false
2026-09-16,5.152000,false
2026-09-17,5.151500,false
2026-09-18,5.156900,false
2026-09-19,5.156900,true
2026-09-20,5.156900,true
2026-09-21,5.156900,true


##### Validação final — falha o Job se algo estiver fora das regras

In [0]:
tb = lambda nome: spark.table(f"{CATALOG}.{SILVER}.{nome}")

checks = {
    "filmes duplicados em tb_info_filmes":
        tb("tb_info_filmes").groupBy("id_filme").count().filter("count > 1").count(),
    "filmes duplicados em tb_financeiro_filmes":
        tb("tb_financeiro_filmes").groupBy("id_filme").count().filter("count > 1").count(),
    "filmes duplicados em tb_metricas_engajamento":
        tb("tb_metricas_engajamento").groupBy("id_filme").count().filter("count > 1").count(),
    "notas TMDB/IMDb fora de 0-10":
        tb("tb_metricas_engajamento").filter(
            "nota_media_tmdb NOT BETWEEN 0 AND 10 OR nota_media_imdb NOT BETWEEN 0 AND 10").count(),
    "notas de usuário fora de 0-10":
        tb("tb_avaliacoes_usuarios").filter("nota_usuario NOT BETWEEN 0 AND 10").count(),
    "valores negativos em popularidade/votos":
        tb("tb_metricas_engajamento").filter(
            "popularidade < 0 OR qtd_votos_tmdb < 0 OR qtd_votos_imdb < 0").count(),
    "financeiro com valores <= 0":
        tb("tb_financeiro_filmes").filter("orcamento_usd <= 0 OR receita_usd <= 0").count(),
    "status fora do domínio":
        tb("tb_info_filmes").filter(
            "status_filme NOT IN ('Lançado','Pós-Produção','Em Produção','Planejado',"
            "'Rumores','Cancelado','Não Informado')").count(),
    "comentários vazios não padronizados":
        tb("tb_avaliacoes_usuarios").filter(
            "comentario_usuario IS NULL OR trim(comentario_usuario) = ''").count(),
    "cotações nulas na série":
        tb("tb_cotacao_dolar").filter("cotacao_compra IS NULL").count(),
}

serie = tb("tb_cotacao_dolar").agg(
    F.count("*").alias("n"), F.min("data_cotacao").alias("d0"), F.max("data_cotacao").alias("d1")
).first()
checks["dias faltando na série da cotação"] = (
    (serie["d1"] - serie["d0"]).days + 1 - serie["n"] if serie["n"] else 1
)

erros = [f"{nome}: {qtd}" for nome, qtd in checks.items() if qtd > 0]
for nome, qtd in checks.items():
    print(f"{'OK  ' if qtd == 0 else 'ERRO'} - {nome}: {qtd}")

if erros:
    raise Exception("Validação Silver falhou:\n" + "\n".join(erros))
print("\nValidação Silver OK")


OK   - filmes duplicados em tb_info_filmes: 0
OK   - filmes duplicados em tb_financeiro_filmes: 0
OK   - filmes duplicados em tb_metricas_engajamento: 0
OK   - notas TMDB/IMDb fora de 0-10: 0
OK   - notas de usuário fora de 0-10: 0
OK   - valores negativos em popularidade/votos: 0
OK   - financeiro com valores <= 0: 0
OK   - status fora do domínio: 0
OK   - comentários vazios não padronizados: 0
OK   - cotações nulas na série: 0
OK   - dias faltando na série da cotação: 0

Validação Silver OK
